# 04 — Model Training & Evaluation

**Prerequisite**: `notebooks/03_clustering.ipynb` must be complete.
The following files must exist:
- `data/processed/zone_hour_grid.parquet`
- `data/processed/zone_day_grid.parquet`
- `data/processed/cis_table.parquet`

**What happens here**:
1. Load aggregated zone×time grids
2. Apply time-based train/test split (train ≤ 2024-02-29 / test ≥ 2024-03-01)
3. Assert no-future-leakage (hard error if violated)
4. Train XGBoost, LightGBM, CatBoost on both hour and day resolutions
5. Evaluate each model: MAE, RMSE, Precision@K, NDCG@10 vs frequency baseline
6. Select winner by NDCG@10 and update configs/model.yaml
7. Save all checkpoints

**Files saved**:
- `checkpoints/{model}_{resolution}_{timestamp}/` — one folder per trained model
- `data/outputs/eval_{timestamp}.json` — full evaluation results
- `configs/model.yaml` — updated with winner

## Cell 1 — Environment setup
**Expected output**: `Project root: ...GridLock R2`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\USER\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Verify prerequisite files exist
**Expected output**: All 3 files found with sizes printed.

In [3]:
required_files = [
    PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'zone_day_grid.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet',
]

all_ok = True
for f in required_files:
    if f.exists():
        print(f'  ✓  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)')
    else:
        print(f'  ✗  MISSING: {f.name} — run 03_clustering.ipynb first!')
        all_ok = False

if not all_ok:
    raise FileNotFoundError('One or more prerequisite files are missing. See above.')
print('\nAll prerequisite files found. Proceed.')

  ✓  zone_hour_grid.parquet  (359.2 KB)
  ✓  zone_day_grid.parquet  (136.0 KB)
  ✓  cis_table.parquet  (10.7 KB)

All prerequisite files found. Proceed.


## Cell 4 — Quick data preview
**What this cell does**: Load and preview both grids before training.  
**Expected output**: Shape and column list for hour and day grids.

In [4]:
import pandas as pd

hour_df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet')
day_df  = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'zone_day_grid.parquet')
cis_df  = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet')

print(f'Zone×Hour grid : {hour_df.shape}  |  cols: {list(hour_df.columns)}')
print(f'Zone×Day  grid : {day_df.shape}   |  cols: {list(day_df.columns)}')
print(f'CIS table      : {cis_df.shape}')
print(f'\nHour target stats:')
print(hour_df['zone_hour_violation_count'].describe())
print(f'\nDay target stats:')
print(day_df['zone_day_violation_count'].describe())

Zone×Hour grid : (26354, 24)  |  cols: ['zone_id', 'date', 'hour_of_day', 'zone_hour_violation_count', 'fraction_at_junction', 'dominant_violation_type', 'dominant_vehicle_type', 'violation_type_primary_encoded', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded', 'data_sent_to_scita_mean', 'is_weekend', 'day_of_week', 'month', 'week_of_year', 'quarter', 'is_month_start', 'is_month_end', 'rolling_7d_count', 'rolling_std_7d', 'lag_24h', 'lag_7d', 'violation_count_lag_1h']
Zone×Day  grid : (8246, 23)   |  cols: ['zone_id', 'date', 'zone_day_violation_count', 'fraction_at_junction', 'dominant_violation_type', 'dominant_vehicle_type', 'violation_type_primary_encoded', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded', 'data_sent_to_scita_mean', 'is_weekend', 'day_of_week', 'month', 'week_of_year', 'quarter', 'is_month_start', 'is_month_end', 'rolling_7d_count', 'rolling_std_7d', 'lag_24h', 'lag_7d', 'violation_count_lag_1h']
CIS table      : (140, 9)

Hour 

## Cell 5 — Run full training pipeline
**What this cell does**: Calls `src/training/train.run_training()` which:
- Applies time-based split (train ≤ 2024-02-29 / test ≥ 2024-03-01)
- Asserts no-future-leakage (hard error if violated)
- Trains XGBoost, LightGBM, CatBoost on BOTH hour and day resolutions (6 runs total)
- Evaluates each run: MAE, RMSE, Precision@5/10, NDCG@5/10 vs baseline
- Saves checkpoints

**Expected runtime**: ~5–10 minutes total (LightGBM fastest, CatBoost slowest).

**Expected output**: Training logs per model, then comparison table, then winner.

In [5]:
from src.training.train import run_training

results = run_training(project_root=PROJECT_ROOT)

print('\n=== Training complete ===')
print(f"Winner: {results['winner']['run']}")
print(f"  NDCG@10     : {results['winner']['NDCG@10']:.4f}")
print(f"  Precision@10: {results['winner']['Precision@10']:.4f}")
print(f"  MAE         : {results['winner']['MAE']:.4f}")

16:11:22 | INFO     | Configs loaded | features.yaml hash: 3a10a4684c62c1dc...
16:11:22 | INFO     | Training run started | timestamp=20260620_104122 | seed=42
16:11:22 | INFO     | CIS table loaded: 140 zones


Training all models:   0%|          | 0/6 [00:00<?, ?run/s]

16:11:23 | INFO     | 
  Training: XGBOOST | resolution=hour
  Target: zone_hour_violation_count | Features: 26
16:11:23 | INFO     | Split: train=19,870 rows (2023-11-09 → 2024-02-29) | test=6,484 rows (2024-03-01 → 2024-04-08) | Leakage guard: PASSED ✓
16:11:23 | INFO     | Computing zone aggregate features (train-only) ...
16:11:23 | INFO     | Zone aggregates computed for 140 training zones | zone_mean_count range: [1.10, 65.32]
16:11:23 | INFO     | X_train: (19870, 26)  y_train mean=10.30 max=284
16:11:23 | INFO     | X_val:   (6484, 26)    y_val mean=9.82 max=266



Fitting xgboost (hour):   0%|          | 0/1 [00:00<?, ?model/s]

16:11:24 | INFO     |   XGBoost early-stop: best_round=61 | final_val_rmse=10.0402


...

16:12:10 | INFO     |   CatBoost early-stop: best_round=97 | final_val_rmse=27.4746



Fitting catboost (day): 100%|██████████| 1/1 [00:00<00:00,  2.72model/s]
                                                                        

16:12:10 | INFO     | === Evaluating: catboost | day | target='zone_day_violation_count' ===
16:12:10 | INFO     | [catboost/day] MAE=10.8560  RMSE=27.3577
16:12:10 | INFO     | Naive mean-per-zone baseline: MAE=11.7889  RMSE=36.2259
16:12:10 | INFO     |   BEATS naive baseline: MAE 10.8560 vs 11.7889 (+7.9% improvement)
16:12:10 | INFO     | Relevance assigned: grade=2 → 35 zones | grade=1 → 35 zones | grade=0 → 67 zones (q50=36.0, q75=113.0)
16:12:10 | INFO     |   [catboost] NDCG@5=1.0000  Precision@5=1.0000
16:12:10 | INFO     |   [catboost] NDCG@10=1.0000  Precision@10=1.0000
16:12:10 | INFO     | Frequency baseline: 140 zones | top-3 zones: {2: 185578.5, 7: 1537.732546, 4: 160.647921}
16:12:10 | INFO     |   [freq-baseline] NDCG@5=1.0000  Precision@5=1.0000
16:12:10 | INFO     |   [freq-baseline] NDCG@10=1.0000  Precision@10=1.0000
16:12:10 | WARNING  |   [catboost] does NOT beat freq ranker on NDCG.
16:12:10 | INFO     |   [catboost] Computing per-hour ranking metrics ...
16:12:

Training all models: 100%|██████████| 6/6 [00:47<00:00,  7.99s/run]

16:12:11 | INFO     | 
  MODEL COMPARISON SUMMARY
16:12:11 | INFO     |   xgboost_hour                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.6808
16:12:11 | INFO     |   lightgbm_hour                  | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.5748
16:12:11 | INFO     |   catboost_hour                  | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.6522
16:12:11 | INFO     |   xgboost_day                    | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=9.6279
16:12:11 | INFO     |   lightgbm_day                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=9.5174
16:12:11 | INFO     |   catboost_day                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=10.8560
16:12:11 | INFO     | 
  🏆 WINNER: lightgbm_hour | NDCG@10=1.0000 | Precision@10=1.0000 | MAE=4.5748
16:12:11 | INFO     | Eval results saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\outputs\eval_20260620_104122.json'
16:12:11 | INFO     | model.yaml winner section updated → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\configs

## Cell 6 — Comparison table
**What this cell does**: Print a formatted table of all 6 model runs ranked by NDCG@10.  
**Expected output**: Table with 6 rows (3 models × 2 resolutions), sorted by NDCG@10.

In [6]:
import pandas as pd

comparison_df = pd.DataFrame(results['comparison_table'])
comparison_df = comparison_df.sort_values('NDCG@10', ascending=False).reset_index(drop=True)

print('=== Model Comparison (sorted by NDCG@10) ===')
print(comparison_df[['run', 'NDCG@10', 'Precision@10', 'MAE', 'RMSE']].to_string(index=False))
print(f"\nWinner → {results['winner']['run']}")

=== Model Comparison (sorted by NDCG@10) ===
          run  NDCG@10  Precision@10       MAE      RMSE
lightgbm_hour      1.0           1.0  4.574786  9.970094
catboost_hour      1.0           1.0  4.652203 10.063377
 xgboost_hour      1.0           1.0  4.680788 10.016481
 lightgbm_day      1.0           1.0  9.517388 24.834146
  xgboost_day      1.0           1.0  9.627862 25.346600
 catboost_day      1.0           1.0 10.855957 27.357727

Winner → lightgbm_hour


## Cell 7 — Baseline vs ML comparison
**What this cell does**: For each model, show whether it beats the frequency ranker baseline on NDCG@10 and Precision@10.  
**Expected output**: ✓ or ✗ for each model vs baseline.

In [7]:
print('=== Baseline Beat Report ===')
print(f'{"Run":<35} {"Beats NDCG":>12} {"Beats Prec":>12}')
print('-' * 62)
for row in results['comparison_table']:
    ndcg_ok = '✓' if row['beats_baseline']['ndcg']      else '✗'
    prec_ok = '✓' if row['beats_baseline']['precision'] else '✗'
    print(f"{row['run']:<35} {ndcg_ok:>12} {prec_ok:>12}")

=== Baseline Beat Report ===
Run                                   Beats NDCG   Beats Prec
--------------------------------------------------------------
lightgbm_hour                                  ✗            ✗
catboost_hour                                  ✗            ✗
xgboost_hour                                   ✗            ✗
lightgbm_day                                   ✗            ✗
xgboost_day                                    ✗            ✗
catboost_day                                   ✗            ✗


## Cell 8 — Checkpoint verification
**What this cell does**: Verify all checkpoint folders were created and contain expected files.  
**Expected output**: List of saved checkpoint directories.

In [8]:
from pathlib import Path

ckpt_root = PROJECT_ROOT / 'checkpoints'
print(f'Checkpoint directory: {ckpt_root}')
print()

for folder in sorted(ckpt_root.iterdir()):
    if folder.is_dir():
        files = list(folder.iterdir())
        print(f'  {folder.name}/')
        for f in sorted(files):
            print(f'    {f.name}  ({f.stat().st_size / 1e3:.1f} KB)')

Checkpoint directory: C:\Users\USER\Desktop\GridLock_R2_Transfer\checkpoints

  catboost_day_20260619_155555/
    eval.yaml  (9.1 KB)
    features.yaml  (9.5 KB)
    label_encoders.pkl  (2.2 KB)
    model.cbm  (95.1 KB)
    model.yaml  (9.0 KB)
    training_meta.json  (9.1 KB)
    zone_stats.parquet  (8.2 KB)
  catboost_day_20260619_165718/
    eval.yaml  (9.1 KB)
    features.yaml  (10.7 KB)
    label_encoders.pkl  (2.2 KB)
    model.cbm  (106.1 KB)
    model.yaml  (10.7 KB)
    training_meta.json  (9.4 KB)
    zone_stats.parquet  (8.2 KB)
  catboost_day_20260619_170427/
    eval.yaml  (9.1 KB)
    features.yaml  (10.7 KB)
    label_encoders.pkl  (2.2 KB)
    model.cbm  (99.4 KB)
    model.yaml  (10.7 KB)
    training_meta.json  (9.2 KB)
    zone_stats.parquet  (8.2 KB)
  catboost_day_20260620_053524/
    eval.yaml  (9.1 KB)
    features.yaml  (9.5 KB)
    label_encoders.pkl  (2.2 KB)
    model.cbm  (95.1 KB)
    model.yaml  (11.2 KB)
    training_meta.json  (11.7 KB)
    zone_stats.p

## Summary

**What was done**:
- Time-based split applied (train ≤ 2024-02-29 / test ≥ 2024-03-01)
- No-future-leakage guard: PASSED
- XGBoost, LightGBM, CatBoost trained on zone×hour and zone×day grids
- Each model evaluated on MAE, RMSE, NDCG@10, Precision@10 vs frequency baseline
- Winner selected by highest NDCG@10 (MAE as tiebreaker)

**Files saved**:
- `checkpoints/{model}_{resolution}_{timestamp}/` (6 checkpoint folders)
- `data/outputs/eval_{timestamp}.json`
- `configs/model.yaml` (updated with winner)

**Next step**: `notebooks/05_inference.ipynb` — load the winning model and generate enforcement priority rankings

In [9]:
print('=== Training Summary ===')
print(f"  Timestamp     : {results['training_timestamp']}")
print(f"  Winner model  : {results['winner']['model']}")
print(f"  Winner resol. : {results['winner']['resolution']}")
print(f"  NDCG@10       : {results['winner']['NDCG@10']:.4f}")
print(f"  Precision@10  : {results['winner']['Precision@10']:.4f}")
print(f"  MAE           : {results['winner']['MAE']:.4f}")
print(f"  RMSE          : {results['winner']['RMSE']:.4f}")
print(f"  Eval output   : data/outputs/eval_{results['training_timestamp']}.json")
print(f"  Next notebook : notebooks/05_inference.ipynb")

=== Training Summary ===
  Timestamp     : 20260620_104122
  Winner model  : lightgbm
  Winner resol. : hour
  NDCG@10       : 1.0000
  Precision@10  : 1.0000
  MAE           : 4.5748
  RMSE          : 9.9701
  Eval output   : data/outputs/eval_20260620_104122.json
  Next notebook : notebooks/05_inference.ipynb
